# Data Preprocessing for BiLSTM

**AI Learning Assistant** — This notebook documents every preprocessing step applied to the cleaned student emotion dataset before BiLSTM model training.

**Input:** `dataset/student_emotions_cleaned.csv`

**Outputs:**
- `dataset/processed_full.csv`
- `dataset/train.csv`, `dataset/validation.csv`, `dataset/test.csv`
- `models/label_encoder.joblib`

> **Note:** No tokenization, model training, or Streamlit UI is created in this module.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.preprocess import (
    ensure_nltk_resources,
    load_cleaned_dataset,
    preprocess_dataframe,
    encode_labels,
    stratified_train_val_test_split,
    save_processed_datasets,
    print_preprocessing_summary,
    preprocess_text,
    to_lowercase,
    expand_contractions,
    remove_urls,
    remove_html_tags,
    remove_emails,
    remove_numbers,
    remove_punctuation,
    remove_special_characters,
    normalize_whitespace,
)

ensure_nltk_resources()

## 2. Load the Cleaned Dataset

The cleaned dataset was produced by the Dataset Collection & Exploration module. It contains deduplicated records with mapped target emotions.

In [ ]:
df = load_cleaned_dataset()
print(f"Records: {len(df):,}")
df.head()

## 3. Preprocessing Pipeline Overview

The BiLSTM pipeline applies the following steps **in order**:

| Step | Function | Purpose |
|------|----------|--------|
| 1 | `to_lowercase` | Normalize casing |
| 2 | `expand_contractions` | Expand *don't* → *do not*, etc. |
| 3 | `remove_urls` | Strip web links |
| 4 | `remove_html_tags` | Strip HTML markup |
| 5 | `remove_emails` | Strip email addresses |
| 6 | `remove_numbers` | Remove numeric tokens |
| 7 | `remove_punctuation` | Remove punctuation marks |
| 8 | `remove_special_characters` | Keep only alphabetic characters |
| 9 | `normalize_whitespace` | Collapse extra spaces |
| 10 | `remove_stopwords` | Remove NLTK English stopwords |
| 11 | `lemmatize_tokens` | Reduce words to base form (WordNet) |

The original `text` column is **never modified**. Results are stored in `processed_text`.

### Step-by-Step Example

Below we walk through each step on a single sample to show the transformation.

In [ ]:
sample = df.iloc[4]["text"]
print("Original:", sample)

step = to_lowercase(sample)
print("\n1. Lowercase:", step)

step = expand_contractions(step)
print("2. Contractions:", step)

step = remove_urls(step)
print("3. URLs removed:", step)

step = remove_html_tags(step)
print("4. HTML removed:", step)

step = remove_emails(step)
print("5. Emails removed:", step)

step = remove_numbers(step)
print("6. Numbers removed:", step)

step = remove_punctuation(step)
print("7. Punctuation removed:", step)

step = remove_special_characters(step)
print("8. Special chars removed:", step)

step = normalize_whitespace(step)
print("9. Whitespace normalized:", step)

print("\n10-11. Full pipeline (stopwords + lemmatization):")
print(preprocess_text(sample))

## 4. Apply Preprocessing to the Full Dataset

In [ ]:
processed_df = preprocess_dataframe(df)
processed_df[["text", "processed_text", "target_emotion"]].head(10)

## 5. Encode Emotion Labels

String emotion labels are converted to integers using `sklearn.preprocessing.LabelEncoder`. The fitted encoder is saved to `models/label_encoder.joblib` for inference.

In [ ]:
encoded_df, encoder = encode_labels(processed_df)

print("Class mapping:")
for idx, name in enumerate(encoder.classes_):
    print(f"  {idx} -> {name}")

encoded_df[["text", "processed_text", "target_emotion", "label"]].head()

## 6. Stratified Train / Validation / Test Split

The dataset is split **80% / 10% / 10%** using stratified sampling on the encoded label column to preserve class proportions across splits.

In [ ]:
train_df, val_df, test_df = stratified_train_val_test_split(encoded_df)

print(f"Train: {len(train_df):,}  |  Validation: {len(val_df):,}  |  Test: {len(test_df):,}")

## 7. Save Processed Datasets

In [ ]:
paths = save_processed_datasets(encoded_df, train_df, val_df, test_df)
for name, path in paths.items():
    print(f"{name}: {path}")

## 8. Summary Report

In [ ]:
print_preprocessing_summary(encoded_df, encoder, train_df, val_df, test_df)